# Sensitivity analysis

One-at-a-time (OAT) sensitivity analysis for the SELT(R) microsimulation. Each parameter is varied ±20% from its baseline while holding all others fixed. Because the microsimulation is stochastic, each scenario is run with a fixed seed for reproducibility. The outcome metrics are:

- **$R_0$**: analytical formula — unaffected by stochasticity.
- **Cumulative incidence**: total entries into $I_\text{sub}$ (onset of active TB) over 50 years (single run per scenario, seed fixed). Note: because R0 is close to 2, stochastic extinction can occur in some parameter scenarios — large differences in incidence sometimes reflect extinction rather than smooth parameter dependence.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.model import compute_basic_reproduction_number, run_simulation
from src.parameters import ModelParameters
from src.plotting import plot_tornado

In [ ]:
parameters = ModelParameters()
time_horizon_years = 50.0
fixed_seed = 0

baseline_results = run_simulation(parameters, time_horizon_years=time_horizon_years, seed=fixed_seed)
baseline_r0 = compute_basic_reproduction_number(parameters)
baseline_cumulative_incidence = int(baseline_results["incident_active_tb"].sum())

print(f"Baseline R0: {baseline_r0:.2f}")
print(f"Baseline cumulative incidence ({time_horizon_years:.0f} years): {baseline_cumulative_incidence:,}")

In [ ]:
from dataclasses import asdict

parameter_names = [
    "transmission_rate",
    "fast_progressor_fraction",
    "subclinical_progression_rate",
    "subclinical_relative_infectiousness",
    "care_seeking_rate",
    "test_sensitivity",
    "linkage_to_treatment_prob",
    "self_cure_rate",
    "tb_mortality_rate",
]

low_multiplier = 0.8
high_multiplier = 1.2

low_r0_values = []
high_r0_values = []
low_incidence_values = []
high_incidence_values = []

for parameter_name in parameter_names:
    base_values = asdict(parameters)
    low_value = base_values[parameter_name] * low_multiplier
    high_value = base_values[parameter_name] * high_multiplier

    low_parameters = ModelParameters(**{**base_values, parameter_name: low_value})
    high_parameters = ModelParameters(**{**base_values, parameter_name: high_value})

    # Fixed seed so the only variation is from the parameter change, not RNG
    low_results = run_simulation(low_parameters, time_horizon_years=time_horizon_years, seed=fixed_seed)
    high_results = run_simulation(high_parameters, time_horizon_years=time_horizon_years, seed=fixed_seed)

    low_r0_values.append(compute_basic_reproduction_number(low_parameters))
    high_r0_values.append(compute_basic_reproduction_number(high_parameters))

    low_incidence_values.append(int(low_results["incident_active_tb"].sum()))
    high_incidence_values.append(int(high_results["incident_active_tb"].sum()))

print("Sensitivity runs complete.")

In [ ]:
plot_tornado(
    parameter_names=parameter_names,
    low_values=np.array(low_r0_values),
    high_values=np.array(high_r0_values),
    baseline_value=baseline_r0,
    x_label="R0",
)
plt.title("One-at-a-time sensitivity: R0")
plt.show()

plot_tornado(
    parameter_names=parameter_names,
    low_values=np.array(low_incidence_values),
    high_values=np.array(high_incidence_values),
    baseline_value=baseline_cumulative_incidence,
    x_label=f"Cumulative incidence ({time_horizon_years:.0f} years)",
)
plt.title("One-at-a-time sensitivity: cumulative incidence (microsimulation)")
plt.show()